In [64]:
import os, json, time
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [65]:
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
print("Pageindex key loaded: ", "TRUE" if PAGEINDEX_API_KEY else "FALSE")

Pageindex key loaded:  TRUE


In [66]:
from pageindex import PageIndexClient
from langchain_ollama import ChatOllama, chat_models

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
ollama_client = ChatOllama(model=os.getenv("OLLAMA_MODEL"), format="json")

print("Pageindex client initialized: ", "TRUE" if pi_client else "FALSE")
print("Ollama client initialized: ", "TRUE" if ollama_client else "FALSE")

Pageindex client initialized:  TRUE
Ollama client initialized:  TRUE


In [67]:
#-----Upload a document to PageIndex-----
# Replace with your own document path
PDF_FILE_PATH = "../data/pdf_file/ReactJS_Interview_Prep.md.pdf"
print(f"Uploading document: {PDF_FILE_PATH} to PageIndex...")
result = pi_client.submit_document(PDF_FILE_PATH)
print("Document uploaded successfully. Document ID:", result["doc_id"])

Uploading document: ../data/pdf_file/ReactJS_Interview_Prep.md.pdf to PageIndex...


PageIndexAPIError: Failed to submit document: {"detail":"InsufficientCredits"}

In [ ]:
print("Building tree index for the uploaded document...")
print("This runs once per document - the index is cached for future queries.")

while True:
    status_result = pi_client.get_document(result["doc_id"])
    status = status_result.get("status")
    print(f"Document status: {status}")
    if status == "completed":
        print("Tree index ready for the document.")
        break
    elif status == "failed":
        print("Document processing failed. Please check the document and try again.")
        break

    time.sleep(5)  # Wait for 5 seconds before checking the status again

Building tree index for the uploaded document...
This runs once per document - the index is cached for future queries.
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: processing
Document status: completed
Tree index ready for the document.


In [ ]:
# --- Fetch the full tree index for the document ---
print("Fetching the full tree index for the document...")
tree_result = pi_client.get_tree(result["doc_id"], node_summary = True)
pageindex_tree = tree_result.get("result", [])
print(f"Top level section: {len(pageindex_tree)}")
print("Raw tree (first node): ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Fetching the full tree index for the document...
Top level section: 21
Raw tree (first node): 
{
  "title": "TABLE OF CONTENTS",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# TABLE OF CONTENTS\n\n1. React Fundamentals & JSX\n2. Components, Props & Composition\n3. State Management with useState\n4. Component Lifecycle & useEffect\n5. Event Handling & Forms\n6. Conditional Rendering & Lists\n7. useRef & DOM Access\n8. useContext & the Context API\n9. useReducer & Complex State Logic\n10. Performance Optimization \u2014 useMemo, useCallback, React.memo\n11. Custom Hooks\n12. The Virtual DOM & Reconciliation\n13. React Router\n14. State Management Libraries (Redux/Zustand)\n15. Data Fetching & Async Patterns\n16. Forms & Validation Libraries\n17. Testing React Components\n18. Error Boundaries & Error Handling\n19. React Performance & Rendering Internals\n20. Modern React Patterns (Suspense, Server Components, Concurrent Features)\n",
  "text": "# TABLE OF CONTENTS\n\n1. React Fun

In [ ]:
# --- Pretty print the tree structure ---
def print_tree(nodes, indent=0):
    """Recursively prints the tree titles for a visual representation of the tree structure."""
    for node in nodes:
        prefix = " " * indent + ("- " if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 2)

print("Full document tree structure:")
print_tree(pageindex_tree)

Full document tree structure:
[0000] TABLE OF CONTENTS (p.1)
[0001] 1. React Fundamentals & JSX (p.1)
[0002] 2. Components, Props & Composition (p.4)
[0003] 3. State Management with useState (p.7)
  - [0004] Q3. Show why mutating an object/array stored in state directly does NOT trigger a re-render, and the immutable-update fix. (p.8)
[0005] 4. Component Lifecycle & useEffect (p.9)
[0006] 5. Event Handling & Forms (p.12)
[0007] 6. Conditional Rendering & Lists (p.14)
[0008] 7. useRef & DOM Access (p.17)
[0009] 8. useContext & the Context API (p.20)
  - [0010] Q1. What problem does Context solve, and show the "prop drilling" anti-pattern it eliminates. (p.20)
  - [0011] Q2. Show creating and providing a Context with a custom Provider component encapsulating both state AND the update logic. (p.20)
[0012] 9. useReducer & Complex State Logic (p.23)
  - [0013] Q4. What is the DANGER of OVER-using useMemo/ useCallback everywhere, and show a case where it's actually COUNTERPRODUCTIVE? (p.28)


In [ ]:
# --- Count the number of nodes in the tree ---
def count_nodes(nodes):
    total = len(nodes)
    for node in nodes:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])
    return total

node_count = count_nodes(pageindex_tree)
print(f"Total number of nodes in the tree: {node_count}")

Total number of nodes in the tree: 32


# Page index retrival

In [ ]:
import json
import re
from langchain_core.messages import HumanMessage


def parse_json_response(response) -> dict:
    """Extract and parse JSON from a LangChain model response."""
    content = response.content

    if isinstance(content, list):
        content = "".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in content
        )

    content = str(content).strip()

    if not content:
        raise ValueError(
            f"Ollama returned an empty response. Raw response: {response!r}"
        )

    # Remove optional Markdown code fences.
    content = re.sub(r"^```(?:json)?\s*", "", content, flags=re.IGNORECASE)
    content = re.sub(r"\s*```$", "", content).strip()

    # Parse the first JSON object if the model added extra text.
    match = re.search(r"\{.*\}", content, flags=re.DOTALL)
    if not match:
        raise ValueError(f"Ollama did not return JSON. Response: {content!r}")

    try:
        result = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON returned by Ollama: {content!r}"
        ) from exc

    if not isinstance(result, dict):
        raise ValueError(f"Expected a JSON object, got: {type(result).__name__}")

    return result


def search_tree_with_llm(query: str, tree: list) -> dict:
    def compress_tree(nodes):
        compressed = []

        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                "page": node.get("page_index", "?"),
                "summary": node.get("summary", "")[:200],
            }

            if node.get("nodes"):
                entry["children"] = compress_tree(node["nodes"])

            compressed.append(entry)

        return compressed

    prompt = f"""
Identify the most relevant document-tree nodes for this query.

Query:
{query}

Document tree:
{json.dumps(compress_tree(tree), indent=2)}

Return ONLY valid JSON:
{{
  "thinking": "Brief explanation",
  "node_list": ["node_id"]
}}
"""

    response = ollama_client.invoke([HumanMessage(content=prompt)])
    result = parse_json_response(response)

    result.setdefault("thinking", "")
    result.setdefault("node_list", [])

    return result



In [68]:
# --- Test with a sample query ---
sample_query = "What are the key ReactJS interview questions?"
print(f"Searching the tree for query: '{sample_query}'")
llm_result = search_tree_with_llm(sample_query, pageindex_tree)
print("LLM reasoning and node selection:")
print(json.dumps(llm_result, indent=2))

Searching the tree for query: 'What are the key ReactJS interview questions?'


ValueError: Ollama returned an empty response. Raw response: AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'Gemma4', 'created_at': '2026-09-04T16:28:29.498479Z', 'done': True, 'done_reason': 'length', 'total_duration': 55242793958, 'load_duration': 4421292, 'prompt_eval_count': 2846, 'prompt_eval_duration': 9135262000, 'eval_count': 1250, 'eval_duration': 46050700000, 'logprobs': None, 'model_name': 'Gemma4', 'model_provider': 'ollama'}, id='lc_run--01a06d3f-35ec-7170-b19d-7feef117dc1d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2846, 'output_tokens': 1250, 'total_tokens': 4096})